#  Outline

2022/05/19- 05/22

- [X] Binary search: summary of patterns
- [X] Time/space complexity. Usually O(log n)
- [X] Typical LeetCode examples
- [X] My summary
- [X] Practical applications

Reference links:
- [leetcode cookbook](https://books.halfrost.com/leetcode/ChapterTwo/Binary_Search/)
- [宫水三叶 - Sorting wiki](https://github.com/SharingSource/LogicStack-LeetCode/wiki/%E6%8E%92%E5%BA%8F)
- [dowalle/algo](https://github.com/dowalle/algo/blob/main/Algorithm/2-%E7%AE%97%E6%B3%95%E5%9F%BA%E7%A1%80/2-%E4%BA%8C%E5%88%86%E6%B3%95.md)

# Key Concepts


- [X] Summary of patterns

Whenever you need to find an index or element in a collection that satisfies some condition, you should consider binary search.

If the collection is unordered, we can always sort it before applying binary search.

'Minimize the maximum value' is a phrasing commonly seen in binary search problems.

- One trick for analyzing binary search is: avoid using else, and instead spell out every case with else if -- this makes all the details explicit.
- Also, when computing mid you need to guard against overflow. In the code, `left + (right - left) // 2` gives the same result as `(left + right) // 2`, but effectively prevents overflow when left and right are large and added directly.

Approach:
1. Determine the range -- the maximum and minimum values.
2. Binary-search by guessing, and check whether the guess satisfies the condition possible(guess). **This turns a satisfiability problem into a decision problem**, which greatly reduces the complexity!
3. If you're not sure how to write the condition for updating the left/right indices, first list out the conditions that are satisfied, then discuss the unsatisfied cases one by one.
4. If you're not sure how to compute mid, imagine the smallest possible window and check whether mid should round left or right, and whether that could cause an infinite loop.


A few points that need special attention

- When the window shrinks to its minimum, if the target id exists,
    - the relationship between mid and left/right
    - the relationship between left and right
- What is the condition being tested? How do you decide whether to move left or right next? When the condition isn't satisfied, how do you update the left/right indices?
    - Whether the right endpoint is closed affects the loop-exit check and how indices are updated (though this is only a matter of form, not substance). Be careful when computing mid too, and watch out for infinite loops.

## Common Cases

Based on the relationship between the condition and the target id, and whether left/right neighbors need to be checked, these can be grouped into the following cases.

### Standard Case

The target id's value equals the condition directly, no need to check left/right neighbors.
- I.e., the loop-exit condition is nums[mid] == target
- When the window shrinks to its minimum,
    - if using closed-left/open-right, i.e. left=right-1, then mid=left=right-1
    - if using closed-left/closed-right, i.e. left=right, then mid=left=right
- The loop-exit condition when it's unsatisfiable
    - If using closed-left/open-right (left index reachable, right index not reachable), exit when left>=right
    - If using closed-left/closed-right, exit when left>right  
- If not satisfied, how the left/right indices are updated: left=mid-1, right=mid+1

In [ ]:
# Standard template

def binarySearch(nums, target):
    """
    :type nums: List[int]
    :type target: int
    :rtype: int
    """
    if len(nums) == 0:
        return -1

    left, right = 0, len(nums) - 1 # closed-left, closed-right approach
    
    while left <= right: # exit condition: left is strictly greater than right,
        # even when left equals right, we still compare with the target
        # equivalently: the search condition can be determined without comparing to the element's two sides (or by using specific elements around it).
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1   

### Finding the First Element Equal to target

The target id's value equals the condition, and we need to check that the **left** neighbor doesn't exist or is less than the target value.

- I.e., the loop-exit condition is satisfied when nums[mid] == target and (mid == 0 or nums[mid-1] < target), meaning the final window should contain the two numbers [mid-1,mid], or [0]. For mid=0, this can be decided upfront by checking whether nums[0] equals target.
- When the window shrinks to its minimum,
    - if using closed-left/open-right, i.e. left=right-2, then mid=left+1=right-1
    - if using closed-left/closed-right, i.e. left=right-1, then mid=left+1=right
- Below we discuss the case nums[0] != target -- the loop-exit condition when unsatisfiable
    - If using closed-left/open-right, right represents a value greater than target or a value equal to target that isn't the first occurrence; left>right-2 (<=>) right-left<=1
    - If using closed-left/closed-right, right represents a value greater than or equal to target; right-left<=0
- If not satisfied, how the left/right indices are updated:
    - If using closed-left/open-right: if (nums[mid] > target) or (nums[mid] == target and nums[mid-1] >= target), right=mid
    - If using closed-left/closed-right: if (nums[mid] > target) or (nums[mid] == target and nums[mid-1] >= target), right=mid - 1
    - The left endpoint is updated the same way: if nums[mid] < target, left=mid


In [19]:
# closed-left, open-right
def BinarySearch(nums, target):
    if len(nums) < 1:
        return -1
    if nums[0] == target: # handle the case where the first element equals target separately
        return 0
    left = 0
    right = len(nums)
    while right-left > 1: # exit condition: only one number left in the window
        mid = (right + left) // 2 
        if nums[mid] > target or (nums[mid] == target and nums[mid-1]>=target):
            right = mid # right represents a position strictly greater than the condition
        elif nums[mid] < target:
            left = mid # left represents a value less than target
        else:
            return mid
    return -1

def BinarySearchV2(nums, target):
    if len(nums) < 1:
        return -1
    left = 0
    right = len(nums)
    while left < right: # exit condition: only one number left in the window
        mid = (right + left) // 2 
        if nums[mid] < target:
            left = mid + 1 # left represents a value less than target
        else:
            right = mid
    if left == len(nums) or nums[left] != target:
        return -1
    return left

# closed-left, closed-right
def BinarySearchClosed(nums, target):
    if len(nums) < 1:
        return -1
    if nums[0] == target:
        return 0
    left = 0
    right = len(nums) - 1 # right index is reachable
    while right - left > 0: # exit condition: only one number left in the window
        mid = (right + left) // 2 + 1 # note: add 1 here. For the case right=left+1, this makes mid=left+1; otherwise mid=left, which would cause an infinite loop
        if nums[mid] > target or (nums[mid] == target and nums[mid-1]>=target):
            right = mid-1 # right is reachable
        elif nums[mid] < target:
            left = mid
        else:
            return mid
    return -1

In [21]:
nums = [1,1,1,3,4,5,5,5,6,6,6,7,8]
target = 8
print(BinarySearch(nums, target))
print(BinarySearchV2(nums, target))
print(BinarySearchClosed(nums, target))

12
12
12


### Finding the Last Element Equal to target

The target id's value equals the condition, and we need to check that the **right neighbor** doesn't exist or is greater than the target value.

- I.e., the loop-exit condition is satisfied when nums[mid] == target and (mid == len(nums)-1 or nums[mid+1] > target), meaning the final window should contain the two numbers [mid,mid+1], or [len(nums)-1]. For mid=len(nums)-1, this can be decided upfront by checking whether nums[-1] equals target.
- When the window shrinks to its minimum,
    - if using closed-left/open-right, i.e. left=right-2, then mid=left=right-2
    - if using closed-left/closed-right, i.e. left=right-1, then mid=left=right-1
- Below we discuss the case nums[-1] != target -- the loop-exit condition when unsatisfiable
    - If using closed-left/open-right, right represents a value greater than target that is **not the first** such value; since the smallest window is left=right-2, the exit condition is left>right-2 (<=>) right < left + 2 or right <= left + 1
    - If using closed-left/closed-right, right represents a value greater than target; similarly, right-left<=0 or right < left + 1
- If not satisfied, how the left/right indices are updated:
    - If using closed-left/open-right: 
        - if (nums[mid] > target) right = mid + 1
        - if (nums[mid] < target) left = mid + 1
        - if (nums[mid] == target and nums[mid+1] <= target) left=mid+1
    - If using closed-left/closed-right: 
        - if (nums[mid] > target) right = mid
        - if (nums[mid] < target) left = mid + 1
        - if (nums[mid] == target and nums[mid+1] <= target), left = mid + 1


In [29]:
# closed-left, open-right
def BinarySearch(nums, target):
    if len(nums) < 1:
        return -1
    left = 0
    right = len(nums)
    while left < right: # exit condition: only one number left in the window
        mid = (right + left) // 2
        if nums[mid] > target:
            right = mid
        else:
            left = mid + 1
    if nums[left-1] != target:
        return -1
    return left - 1

# closed-left, closed-right
def BinarySearchClosed(nums, target):
    if len(nums) < 1:
        return -1
    if nums[-1] == target:
        return len(nums)-1
    left = 0
    right = len(nums) - 1 # right index is reachable
    while right - left > 0: # exit condition: only one number left in the window
        mid = (right + left) // 2 # note: no need to add 1 here -- for the case right=left+1, mid=left works fine
        if nums[mid] < target or (nums[mid] == target and nums[mid+1]<=target):
            left = mid + 1
        elif nums[mid] > target:
            right = mid # no +1 here, right represents a value greater than target
        else:
            return mid
    return -1

In [34]:
nums = [1,1,1,3,4,5,5,5,6,6,6,7,8]
target = 3
print(BinarySearch(nums, target))
print(BinarySearchClosed(nums, target))

3
3


### Finding the First Element Greater Than or Equal to target

Overall the approach is the same as finding the first element equal to target, with a change in the condition.

The target id's value is **greater than or equal to** the condition, and we need to check that the **left** neighbor doesn't exist or is less than the target value.

- I.e., the loop-exit condition is satisfied when nums[mid] >= target and (mid == 0 or nums[mid-1] < target), meaning the final window should contain the two numbers [mid-1,mid], or [0]. For mid=0, this can be decided upfront by checking whether nums[0] equals target.
The difference from finding the first element equal to target is:
- If not satisfied, how the left/right indices are updated:
    - If using closed-left/open-right: 
        - if nums[mid] < target, left = mid
        - if nums[mid] >= target and nums[mid-1] >= target, right = mid
    - If using closed-left/closed-right: 
        - if nums[mid] < target, left = mid
        - if nums[mid] >= target and nums[mid-1] >= target, right = mid - 1

In [8]:
# closed-left, open-right
def BinarySearch(nums, target):
    if len(nums) < 1:
        return -1
    if nums[0] == target: # handle the case where the first element equals target separately
        return 0
    left = 0
    right = len(nums)
    while right-left > 1: # exit condition: only one number left in the window
        mid = (right + left) // 2 
        if nums[mid] < target:
            left = mid # left represents a value less than target
        elif nums[mid-1] >= target:
            right = mid 
        else:
            return mid
    return -1

def BinarySearchV2(nums, target):
    if not nums:
        return -1
    l = 0
    r = len(nums)
    while l < r:
        mid = l + (r-l)//2
        if nums[mid] >= target:
            r = mid
        else:
            l = mid + 1
    return r
    

# closed-left, closed-right
def BinarySearchClosed(nums, target):
    if len(nums) < 1:
        return -1
    if nums[0] == target:
        return 0
    left = 0
    right = len(nums) - 1
    while right - left > 0: 
        mid = (right + left) // 2 + 1 # note: add 1 here. For the case right=left+1, this makes mid=left+1; otherwise mid=left, which would cause an infinite loop
        if nums[mid] < target:
            left = mid 
        elif nums[mid-1] >= target:
            right = mid - 1
        else:
            return mid
    return -1

In [11]:
nums = [1,1,1,3,5,5,5,5,6,6,6,7,8]
target = 2
print(BinarySearch(nums, target))
print(BinarySearchV2(nums, target))
print(BinarySearchClosed(nums, target))

3
3
3


### Finding the Last Element Less Than target

Overall the approach is the same as finding the last element equal to target, with a change in the condition.

The target id's value is **less than** the condition, and we need to check that the **right** neighbor doesn't exist or is greater than or equal to the target value.

- I.e., the loop-exit condition is satisfied when nums[mid] < target and (mid == len(nums)-1 or nums[mid+1] >= target), meaning the final window should contain the two numbers [mid,mid+1], or [len(nums)-1]. For mid=len(nums)-1, this can be decided upfront by checking whether nums[-1] is less than target.

The difference from finding the last element equal to target is:
- If not satisfied, how the left/right indices are updated:
    - If using closed-left/open-right: 
        - if (nums[mid] >= target) right = mid + 1
        - if (nums[mid] < target and nums[mid+1] < target) left = mid + 1
    - If using closed-left/closed-right: 
        - if (nums[mid] >= target) right = mid
        - if (nums[mid] < target and nums[mid+1] < target) left = mid + 1

In [22]:
# closed-left, open-right
def BinarySearch(nums, target):
    if len(nums) < 1:
        return -1
    if nums[-1] < target: # handle the case comparing the last element to target separately
        return len(nums)-1
    left = 0
    right = len(nums)
    while right-left > 1: # exit condition: only one number left in the window
        mid = (right + left) // 2 
        if nums[mid] >= target:
            right = mid + 1 # right represents a value strictly greater than target
        elif nums[mid+1] < target:
            left = mid + 1 # left represents a value less than target
        else:
            return mid
    rturn -1

# closed-left, closed-right
def BinarySearchClosed(nums, target):
    if len(nums) < 1:
        return -1
    if nums[-1] < target:
        return len(nums)-1
    left = 0
    right = len(nums) - 1 
    while right - left > 0:
        mid = (right + left) // 2 # note: no need to add 1 here -- for the case right=left+1, mid=left works fine
        if nums[mid] >= target:
            right = mid # no +1 here, right represents a value greater than or equal to target
        elif nums[mid+1] < target:
            left = mid + 1
        else:
            return mid
    return -1

In [23]:
nums = [1,1,1,3,5,5,5,5,6,6,6,7,8]
target = 4
print(BinarySearch(nums, target))
print(BinarySearchClosed(nums, target))

3
3


## Generalizing Binary Search Problems

Application: in concrete algorithm problems, the common scenarios are searching the left boundary and searching the right boundary -- rarely are you asked to search for a single element in isolation.

From [labuladong](https://labuladong.github.io/algo/2/18/28/)

The templates for searching the left and right boundaries are below. The difference between the two algorithm processes is:
- Return value: searching the left boundary returns left; searching the right boundary returns left-1
- When target is found: searching the left boundary sets right=mid; searching the right boundary sets left=mid+1


First, you need to abstract a variable `x` from the problem, a function `f(x)` of `x`, and a target value target. At the same time, x, f(x), and target must satisfy the following conditions:

1. `f(x)` must be a monotonic function of `x` (either monotonically increasing or decreasing is fine).

2. The problem asks you to compute the value of `x` that satisfies the constraint `f(x) == target`.

In [ ]:
# search left boundary
def left_bound(nums, target):
    if not nums:
        return -1
    left = 0
    right = len(nums)
    
    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] == target: 
            # when target is found, shrink the right boundary
            right = mid
        elif nums[mid] < target:
            left = mid + 1
        else # if nums[mid] > target:
            right = mid
    return left

# search right boundary
def right_bound(nums, target):
    if not nums:
        return -1
    left = 0
    right = len(nums)
    
    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] == target: 
            # when target is found, shrink the left boundary
            left = mid + 1
        elif nums[mid] < target:
            left = mid + 1
        else # if nums[mid] > target:
            right = mid
    return left - 1

# generalized template for binary search problems


# function f is monotonic in the variable x
def f(x):
    # ...
    

# main function: find the extremal x subject to f(x) == target
def solution(nums, target):
    if not nums:
        return -1
    # ask yourself: what is the minimum value of the variable x?
    left = ...
    # ask yourself: what is the maximum value of the variable x?
    right = ... + 1 # use an open interval here
    
    while left < right:
        mid = left + (right - left) // 2
        if f(mid) == target: 
            # ask yourself: does the problem want the left boundary or the right boundary?
            # ...
        elif f(mid) < target:
            # ask yourself: how do I make f(x) bigger?
            # ...
        else: # if (f(mid) > target) {
            # ask yourself: how do I make f(x) smaller?
            # ...
    return left # or left - 1


# LeetCode Examples


Using binary search on arrays that are basically ordered. The classic solution works, and variant approaches can be written too. Common problem types: finding the peak in a mountain array, finding the pivot point in a rotated sorted array.
The main difficulty lies in designing the condition -- how to choose left vs. right?

- [(medium)33 Search in Rotated Sorted Array](https://leetcode.cn/problems/search-in-rotated-sorted-array/). Approach: overall the same idea as with a sorted array -- compare with the middle element and decide the next window based on the comparison. The difference is that deciding the next window requires case analysis.
- [(medium)81 Search in Rotated Sorted Array II](https://leetcode.cn/problems/search-in-rotated-sorted-array-ii/). Approach: mostly the same as 33, but because of duplicates, the left and right ends can be equal, in which case you must scan one by one. So in the worst case the time complexity is linear and can't reach log.
- [(medium)153 Find Minimum in Rotated Sorted Array](https://leetcode.cn/problems/find-minimum-in-rotated-sorted-array/) Approach: mostly the same as 33.
- [(hard)154 Find Minimum in Rotated Sorted Array II](https://leetcode.cn/problems/find-minimum-in-rotated-sorted-array-ii/). Approach: combines 153 and 81, with extra handling for when both ends are equal.
- [(medium)162 Find Peak Element](https://leetcode.cn/problems/find-peak-element/). Approach: binary search -- the key is how to choose the next left/right. Insight: always binary-search toward the side that's greater at the boundary; this guarantees the chosen interval always contains a peak, and gradually converges to the peak's position.
- [(medium)540 Single Element in a Sorted Array](https://leetcode.cn/problems/single-element-in-a-sorted-array/). Approach: how to choose left vs right? Check whether the even index and the following odd index are equal. If every number appears exactly twice, the number at an even index must equal the number at the following odd index. If they're equal, go right: left=mid + 1; if not, go left: right = mid
- [(easy)852 Peak Index in a Mountain Array](https://leetcode.cn/problems/peak-index-in-a-mountain-array/)/[(easy)911 Sword Offer II 069 Peak of a Mountain Array](https://leetcode.cn/problems/B1IidL/) Approach: how to choose left vs right? If the middle value is greater than both its left and right neighbors, it's the peak; otherwise, if the middle value is less than its left neighbor, the peak must be on the left; otherwise, if the middle value is less than its right neighbor, the peak must be on the right.
- 🌟🌟[(hard)4 Median of Two Sorted Arrays](https://leetcode.cn/problems/median-of-two-sorted-arrays/)
    - Approach 1: find the k-th smallest number across the two arrays using binary search, eliminating k/2 elements each time.
    - Approach 2: regular binary search; the difficulty is designing the condition and how to choose left vs right. Implement this later when there's time -- there are many, many details to handle.



Max-min problems: minimize the maximum value subject to a condition. The overall approach: turn a value-computation problem into a decision problem. Guess a result within a bounded range, and check whether it can be satisfied.

- 🌟[(medium)287 Find the Duplicate Number](https://leetcode.cn/problems/find-the-duplicate-number/). This problem has multiple solutions -- the fast/slow pointer and binary/bit-manipulation approaches in the editorial look quite clever; study them later. Here we implement the binary search approach: guess a duplicate number, then check whether it satisfies the condition, and choose the left or right interval accordingly.
- [(hard)410 Split Array Largest Sum](https://leetcode.cn/problems/split-array-largest-sum/). Approach: turn this into a decision problem, using binary search to guess the target value. If our guessed value can be achieved by splitting into m segments, then the true minimum is no greater than the guess, so choose the left side; otherwise choose the right side. How do we decide whether it can be achieved with m segments? Use a greedy approach: keep each segment's sum no greater than the target value, and count the minimum number of segments needed. If the number of segments used exceeds m, the target value is too small to be achievable. Otherwise, the true minimum is less than or equal to the target value.
- [(medium)875 Koko Eating Bananas](https://leetcode.cn/problems/koko-eating-bananas/). Approach: same as 410 -- first determine the bounds, then check whether the guessed value satisfies the condition, and how to choose left vs right. The key is designing the check.
- [(medium)1011 Capacity to Ship Packages Within D Days](https://leetcode.cn/problems/capacity-to-ship-packages-within-d-days/). Approach: exactly the same as 410.
- [(medium)1283 Find the Smallest Divisor Given a Threshold](https://leetcode.cn/problems/find-the-smallest-divisor-given-a-threshold/). Approach: same as above. Note one detail: the divisor must be greater than or equal to 1, i.e. it cannot be 0.
- 🌟[(hard)668 Kth Smallest Number in Multiplication Table](https://leetcode.cn/problems/kth-smallest-number-in-multiplication-table/). Approach: given a number, we can compute how many numbers in the multiplication table are less than or equal to it. What we want is the **first** number for which there are k numbers less than or equal to it. Pattern: binary-search-guess + check.


- [(medium)34 Find First and Last Position of Element in Sorted Array](https://leetcode.cn/problems/find-first-and-last-position-of-element-in-sorted-array/). Approach: basically applies the standard template. The difference: when the middle value equals the target, move the index left and right to find the start and end positions.
- [(easy)35 Search Insert Position](https://leetcode.cn/problems/search-insert-position/). Approach: the key is understanding what this position actually means: the first position greater than or equal to the target value. Special case: when the target is greater than all values, return the array length.
- [(easy)367 Valid Perfect Square](https://leetcode.cn/problems/valid-perfect-square/). Basic problem
- [(easy)704 Binary Search](https://leetcode.cn/problems/binary-search/). Basic problem: standard template
- [(easy)744 Find Smallest Letter Greater Than Target](https://leetcode.cn/problems/find-smallest-letter-greater-than-target/). Basic problem: handle the wraparound case, i.e. when the last letter is still smaller than the target, return the first letter.
- [(medium)911 Online Election](https://leetcode.cn/problems/online-election/). Approach: binary search over time points to find the last time point less than or equal to the query time, then return the record of whoever had the most votes at that time. This record can be built for every time point during initialization.
- [(medium)1818 Minimum Absolute Sum Difference](https://leetcode.cn/problems/minimum-absolute-sum-difference/)
    - Approach 1: use the guess-and-check pattern for finding an extremum under a constraint: bound the range, guess the possible minimum sum, then check whether it's achievable. The time complexity is fairly high -- although the outer binary search is O(log n), the inner check is O(n^2), giving O(n^2 log n) overall
    - Approach 2: for each i, determine the maximum improvement it can provide: for a given nums1[i], there are two diffs:
        - the current diff: |nums1[i] - nums2[i]|,
        - the difference to the value in nums1 closest to the corresponding nums2: |nums1[j] - nums2[i]|. To find j, first sort nums1, then use binary search on nums1 to find the value closest to nums2[i].
        The difference between these two is the improvement i can contribute. Return the maximum improvement. Time complexity is O(nlogn): sorting is O(nlogn), iterating over each i is O(n), and finding the smallest diff within each iteration is O(logn), giving O(nlogn) overall
- [(easy)1984 Minimum Difference Between Highest and Lowest of K Scores](https://leetcode.cn/problems/minimum-difference-between-highest-and-lowest-of-k-scores/)
    - Approach 1: sliding window -- sort first, then compare against the current minimum. Time complexity is O(nlogn)
- [(easy)Sword Offer 53 Find Number in Sorted Array I](https://leetcode.cn/problems/zai-pai-xu-shu-zu-zhong-cha-zhao-shu-zi-lcof/). Approach: basic problem, similar to 34.


In [ ]:
# 33 binary search, case analysis
class Solution:
    def search(self, nums: List[int], target: int) -> int:
        left = 0
        right = len(nums) - 1
        while left <= right:
            mid = (left + right) // 2
            if nums[mid] == target:
                return mid
            if nums[left] <= nums[right]: # same as the fully sorted case
                if target < nums[left] or  target > nums[right]: 
                    return -1
                if target == nums[left]:
                    return left
                if target == nums[right]:
                    return right
                if nums[mid] < target: # smaller than the middle value, so move to the smaller side
                    left = mid + 1
                else:
                    right = mid - 1
            # all other cases indicate the array has gone through a reversal.
            elif nums[mid] < target:  # target is greater than the middle value, so next we search the "larger" half
                if nums[mid] <= nums[right] and nums[right] < target: # middle value <= right value means the right side is sorted; right value < target means we need to go left.
                    right = mid - 1
                else: 
                    left = mid + 1
            else: # target is less than the middle value, so next we search the "smaller" half
                if nums[mid] > nums[right] and target <= nums[right]: # middle value > right value means the right side is reversed; when target <= right value, go right.
                    left = mid + 1
                else:
                    right = mid - 1
        return -1

In [ ]:
# 81 case analysis
class Solution:
    def search(self, nums: List[int], target: int) -> bool:
        left = 0
        right = len(nums) - 1
        while left <= right:
            if nums[left] == nums[right]: # when the left and right ends are equal, we can't tell which side to go. We need to "remove" these equal numbers first.
                if nums[left] == target:
                    return True
                while left < right and nums[left] == nums[right]: # keep moving the left/right boundaries while they remain equal.
                    left += 1
                    if nums[left] != nums[right]:
                        break
                    right -= 1
                if left > right:
                    return False
                
            mid = (left + right) // 2
            if nums[mid] == target or nums[left] == target or nums[right] == target:
                return True
            # all cases below: left and right are not equal
            if nums[left] < nums[right]: # same as the fully sorted case
                if target < nums[left] or  target > nums[right]: 
                    return False
                if nums[mid] < target: # smaller than the middle value, so move to the smaller side
                    left = mid + 1
                else:
                    right = mid - 1
            # all other cases indicate the array has gone through a reversal.
            elif nums[mid] < target:  # target is greater than the middle value, so next we search the "larger" half
                if nums[mid] <= nums[right] and nums[right] < target: # middle value <= right value means the right side is sorted; right value < target means we need to go left.
                    right = mid - 1
                else: 
                    left = mid + 1
            else: # target is less than the middle value, so next we search the "smaller" half
                if nums[mid] > nums[right] and target < nums[right]: # middle value > right value means the right side is reversed; when target < right value, go right.
                    left = mid + 1
                else:
                    right = mid - 1
        return False

In [ ]:
# 153 binary search, each step moves toward the interval that may contain the minimum
class Solution:
    def findMin(self, nums: List[int]) -> int:
        left, right = 0, len(nums) - 1
        while left <= right:
            if nums[left] <= nums[right]:
                return nums[left]
            mid = (left + right) // 2
            if nums[mid] < nums[left]:
                right = mid
            else:
                left = mid + 1

In [ ]:
# 154 combines 81 and 153, need to handle the case where left and right are equal
class Solution:
    def findMin(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        while left <= right:
            while left < right and nums[left] == nums[right]:
                left += 1
                if nums[left] != nums[right]:
                    break
                right -=1
            if left >= right:
                return nums[right]
            mid = (left + right) // 2
            if nums[left] < nums[right]: # ordered case
                return nums[left]
            else: # reversed case
                if nums[mid] >= nums[left]:
                    left = mid + 1
                else:
                    right = mid


In [ ]:
# 162 the difference between the two implementations: the first one has an early stop.
# The second implementation is much simplified, but there are quite a few details to handle, such as left<right instead of left<=right
# When updating indices, right = middle instead of right = middle - 1; this guarantees middle+1 is always accessible.
# The way to simplify is to fold all the cases that would need discussion into detail handling instead.

class Solution:
    def findPeakElement(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        while left <= right:
            larger_left = False
            larger_right = False
            mid = (left + right) // 2
            if mid == 0 or nums[mid] > nums[mid - 1]:
                larger_left = True
            if mid == len(nums) - 1 or nums[mid] > nums[mid + 1]:
                larger_right = True
            if larger_left and larger_right:
                return mid # early stop
            if larger_left:
                left = mid + 1
            else:
                right = mid - 1
        return -1


class Solution:
    def findPeakElement(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        while left < right:
            middle = (left + right) // 2
            if nums[middle] < nums[middle + 1]: # always binary-search toward the side greater than the boundary; this guarantees the chosen interval always contains a peak
                left = middle + 1
            else:
                right = middle
        return left


In [ ]:
# 540: the key is determining whether to go left or right, and how to update the indices
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        while left < right: 
            even_mid = (left + right) // 2 # even index
            if even_mid % 2 == 1:
                even_mid -= 1 # if the middle value is odd, we know there's always an even index right before an odd index, since the smallest index is 0, which is even. So -= 1 won't go out of bounds.
            odd_mid = even_mid + 1
            if nums[even_mid] == nums[odd_mid]: # if equal, we know the number that appears only once is on the right.
                left = odd_mid + 1 # start right after the odd index
            else:
                right = even_mid # if not equal, we know the number that appears only once is on the left.
        return nums[left]
    
# simplified version, removes the odd_mid variable
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        while left < right:
            mid = (left + right) // 2
            if mid % 2 == 1:
                mid -= 1
            if nums[mid] == nums[mid + 1]:
                left = mid + 2
            else:
                right = mid
        return nums[left]

In [ ]:
# 852 / Sword Offer II 069
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        left, right = 0, len(arr) - 1 # the diff >= 2
        while left <= right - 2: # always keep at least three numbers in the window
            mid = (left + right) // 2 # mid is strictly greater than left and strictly less than right, so mid-1 and mid+1 are always within the window
            if arr[mid] > arr[mid - 1] and arr[mid] > arr[mid + 1]: # found the peak.
                return mid
            if arr[mid] < arr[mid - 1]: # smaller than the left neighbor, the peak must be on the left
                right = mid
            else: # smaller than the right neighbor, the peak must be on the right
                left = mid
        return -1

In [ ]:
# 4 Approach 1: find the k-th smallest number

class Solution:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:
        # change  this to find the [m+n]/2 smallest number
        len1 = len(nums1)
        len2 = len(nums2)
        k = (len1 + len2 + 1) // 2
        module = (len1 + len2 + 1) % 2
        k_smallest, nums1_new, nums2_new = self.findKthItem(nums1, nums2, k)
        if module == 1:
            kplus1_smallest, _, _ = self.findKthItem(nums1_new, nums2_new, 1)
            return (k_smallest + kplus1_smallest) / 2
        return k_smallest

    def findKthItem(self, nums1, nums2, k):
        # print(nums1, nums2, k)
        if len(nums1) == 0:
            return nums2[k-1], [], nums2[k:]
        if len(nums2) == 0:
            return nums1[k-1], nums1[k:], []
        if k == 1:
            if nums1[0] < nums2[0]:
                return nums1[0], nums1[1:], nums2
            else:
                return nums2[0], nums1, nums2[1:]
        half_k = k // 2
        nums1_id = min(len(nums1), half_k) - 1
        nums2_id = min(len(nums2), half_k) - 1
        if nums1[nums1_id] < nums2[nums2_id]:
            return self.findKthItem(nums1[(nums1_id + 1): ], nums2,  k-nums1_id-1)
        if nums1[nums1_id] == nums2[nums2_id]:
            left_k = k - nums1_id - nums2_id - 2
            if left_k == 0:
                return nums1[nums1_id], nums1[nums1_id+1:], nums2[nums2_id + 1:]
            else:
                return self.findKthItem(nums1[nums1_id+1:], nums2[nums2_id + 1:], left_k)
        return self.findKthItem(nums1, nums2[nums2_id + 1:],  k-nums2_id-1)

In [ ]:
# 287 binary search
class Solution:
    def findDuplicate(self, nums: List[int]) -> int:
        left, right = 0, len(nums) - 1
        while left < right:
            mid = (left + right) // 2 + 1
            cnt = 0
            for num in nums:
                if num < mid:
                    cnt += 1
            if cnt < mid:
                left = mid
            else:
                right = mid - 1
        return left

In [ ]:
# 410 turn into a decision problem, use binary search to guess the target value
class Solution:
    def achievable(self, target, nums, m): # the check function
        cnt = 1
        cur_sum = nums[0]
        for num in nums[1:]:
            cur_sum += num
            if cur_sum > target:
                cur_sum = num
                cnt += 1
        return cnt <= m

    def splitArray(self, nums: List[int], m: int) -> int:
        ttl_sum = sum(nums)
        left = ttl_sum // m
        if ttl_sum % m > 0:
            left += 1
        left = max(left, max(nums)) # find the lower bound
        right = ttl_sum - sum(nums[:m-1]) # find the upper bound
        while left < right:
            mid = (left + right) // 2
            if self.achievable(mid, nums, m): # check whether it's achievable; if so choose the left side, otherwise the right side
                right = mid
            else:
                left = mid + 1
        return left


In [ ]:
# 875 same as 410
class Solution:
    def CanFinish(self, mid, piles, h):
        cnt = 0
        for pile in piles:
            if pile <= mid:
                cnt += 1
            else:
                cur_cnt = pile // mid
                if pile % mid > 0:
                    cur_cnt += 1
                cnt += cur_cnt
        return cnt <= h
        
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        ttl = sum(piles) 
        left = ttl // h
        if ttl % h > 0:
            left += 1
        right = max(piles)
        while left < right:
            mid = (left + right) // 2
            if self.CanFinish(mid, piles, h):
                right = mid
            else:
                left = mid + 1
        return left


In [ ]:
# 1011 same as 410
class Solution:
    def CanShip(self, capacity, weights, days):
        actual_days = 1
        cur_sum = weights[0]
        for weight in weights[1:]:
            cur_sum += weight
            if cur_sum > capacity:
                cur_sum = weight
                actual_days += 1
        return actual_days <= days

    def shipWithinDays(self, weights: List[int], days: int) -> int:
        ttl_weights = sum(weights) 
        left = ttl_weights // days
        if ttl_weights % days > 0:
            left += 1
        left = max(left, max(weights))
        right = ttl_weights
        while left < right:
            mid = (left + right) // 2
            if self.CanShip(mid, weights, days):
                right = mid
            else:
                left = mid + 1
        return left

In [ ]:
# 1283
class Solution:
    def UnderThreshold(self, divisor, nums, threshold):
        actual_sum = 0
        for num in nums:
            actual_sum += (num // divisor)
            if num % divisor > 0:
                actual_sum += 1
        return actual_sum <= threshold
                
    def smallestDivisor(self, nums: List[int], threshold: int) -> int:
        left = max(sum(nums) // threshold, 1) # note here: the divisor must be >= 1.
        right = max(nums)
        while left < right:
            mid = (left + right) // 2
            if self.UnderThreshold(mid, nums, threshold):
                right = mid
            else:
                left = mid + 1
        return left

In [ ]:
# 34
class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        left = 0
        right = len(nums) - 1
        while left <= right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            elif nums[mid] > target:
                right = mid - 1
            else:
                start_id = mid
                while start_id >= 0 and nums[start_id] == target:
                    start_id -= 1
                start_id += 1
                end_id = mid
                while end_id <= len(nums) - 1 and nums[end_id] == target:
                    end_id += 1
                end_id -= 1
                return [start_id, end_id]

        return [-1, -1]

In [ ]:
# 35
class Solution:
    def searchInsert(self, nums: List[int], target: int) -> int:
        # find the id of the first element equal to or larger than the target
        left = 0
        right = len(nums) - 1
        while left <= right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            elif nums[mid] == target:
                return mid
            else:
                if left == right:
                    break
                right = mid
        return left

In [ ]:
# 367 
class Solution:
    def isPerfectSquare(self, num: int) -> bool:
        left = 1
        right = num
        while left <= right:
            mid = (left + right) // 2
            actual_square = mid ** 2
            if actual_square > num:
                right = mid - 1
            elif actual_square == num:
                return True
            else:
                left = mid + 1
        return False

In [ ]:
# 744
class Solution:
    def nextGreatestLetter(self, letters: List[str], target: str) -> str:
        if letters[0] > target or letters[-1] <= target: # handle the wraparound case
            return letters[0]
        left = 0
        right = len(letters) - 1
        while left < right:
            mid = (left + right) // 2
            if letters[mid] <= target:
                left = mid + 1
            else:
                right = mid
        return letters[left]

In [ ]:
# 911 build a max-vote-person record for every time point during initialization. Binary search over time points, then return the record for the matching time point
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        self.persons = persons
        self.times = times
        self.cummu = {} # cumulative vote count for each person up to the current time
        self.max_person = [] # build a max-vote-person record during initialization
        self.max_vote = 0
        self.ttl_len = len(persons)
        for i in range(self.ttl_len):
            person = persons[i]
            self.cummu[person] = self.cummu.get(person, 0) + 1
            if self.cummu[person] >= self.max_vote: # in case of a tie, the person who got the vote most recently leads by default.
                self.max_vote = self.cummu[person]
                self.max_person.append(person)
            else:
                self.max_person.append(self.max_person[-1])

    def q(self, t: int) -> int:
        # first, find the id of the last time smaller or equal to the t
        # use the binary search on self.times
        left = 0
        right = self.ttl_len - 1
        if self.times[right] <= t:
            return self.max_person[right]
        if self.ttl_len == 1:
            return self.max_person[0]
        while left < right:
            mid = (left + right) // 2 + 1
            if self.times[mid] < t:
                left = mid
            elif self.times[mid] == t:
                return self.max_person[mid]
            else:
                right = mid - 1
        # compare and return the result
        return self.max_person[left]



# Your TopVotedCandidate object will be instantiated and called as such:
# obj = TopVotedCandidate(persons, times)
# param_1 = obj.q(t)

In [ ]:
# 1818 Approach 1: use the guess-then-check pattern for finding an extremum under a constraint
class Solution:
    def CanGet(self, reduced_diff, nums1, nums2, diff):
        for i in range(len(diff)): # O(n)
            if diff[i] < reduced_diff:
                continue
            ideal_diff = diff[i] - reduced_diff
            for target_num in range(nums2[i] - ideal_diff, nums2[i] + ideal_diff + 1):  # O(n)
                if target_num in nums1:
                    return True
        return False

    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        diff = [abs(nums1[i] - nums2[i]) for i in range(len(nums1))]
        max_sum = sum(diff)
        right = max_sum
        left = right - max(diff)
        while left < right: # O(log n) complexity
            mid = (left + right) // 2
            if self.CanGet(max_sum-mid, nums1, nums2, diff):
                right = mid
            else:
                left = mid + 1
        return left % (10**9 + 7)

In [ ]:
# 1818 Approach 2: the maximum improvable diff

class Solution:
    def findMostClosed(self, nums, target):
        # in nums find the one with the smallest difference from target and return the diff
        left = 0
        right = len(nums) - 1
        min_diff = abs(nums[0][0] - target)
        while left <= right:
            mid = (left + right) // 2
            if nums[mid][0] == target:
                return 0
            if nums[mid][0] < target:
                if target - nums[mid][0] < min_diff:
                    min_diff = target - nums[mid][0]
                left = mid + 1
            else:
                if nums[mid][0] - target < min_diff:
                    min_diff = nums[mid][0] - target
                right = mid - 1
        return min_diff
            
    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        to_sort_nums1 = [(num, i) for i, num in enumerate(nums1)]
        to_sort_nums1.sort()
        diff = [abs(item[0] - nums2[item[1]]) for item in to_sort_nums1]
        max_sum = sum(diff)
        max_reduced = 0
        for i, item in enumerate(to_sort_nums1):
            if diff[i] <= max_reduced:
                continue # this early continue reduced the runtime from 1388ms to 300ms
            closed_diff = self.findMostClosed(to_sort_nums1, nums2[item[1]])
            if diff[i] - closed_diff > max_reduced:
                max_reduced = diff[i] - closed_diff
        return (max_sum - max_reduced) % (10**9 + 7)

In [ ]:
# 1984 Approach 1: sliding window -- sort, then compare
class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:
        if k == 1:
            return 0
        nums.sort()
        min_diff = nums[k-1] - nums[0]
        for i in range(1, len(nums) - k + 1):
            if nums[i+k-1] - nums[i] < min_diff:
                min_diff = nums[i + k - 1] - nums[i]
        return min_diff

In [ ]:
# Sword Offer 53
class Solution:
    def search(self, nums: List[int], target: int) -> int:
        left = 0
        right = len(nums) - 1
        start_id = -1
        end_id = -1
        while left <= right:
            mid = (left + right) // 2
            if nums[mid] == target:
                start_id = end_id = mid
                while start_id >= 0 and nums[start_id] == target:
                    start_id -= 1
                start_id += 1
                while end_id <= len(nums) - 1 and nums[end_id] == target:
                    end_id += 1
                end_id -= 1
                return end_id - start_id + 1
            elif nums[mid] < target:
                left = mid + 1
            else:
                right = mid - 1
        return 0

In [ ]:
# 668
class Solution:
    def CountNumber(self, goal, m, n):
        cnt = 0
        for i in range(min(m, goal)):
            cnt += min(n, goal // (i + 1))
        return cnt 
    
    def findKthNumber(self, m, n, k):
        left = 1
        right = m * n 
        while left < right:
            mid = (left + right) // 2
            cnt = self.CountNumber(mid, m, n)
            if cnt >= k:
                right = mid
            else:
                left = mid + 1
        return right


# My Summary

- When doing binary search on a rotated sorted array, one thing to keep in mind: after splitting, you need to handle the case where a sub-interval turns out to be fully sorted. The pattern is to carefully enumerate all the cases. See examples 33, 81, 153
- For finding an extremum under a constraint: the pattern is to bound the range, guess a value, check whether it satisfies the condition, then gradually narrow the range.